# Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer (T5)

**Paper:** [arXiv:1910.10683](https://arxiv.org/abs/1910.10683)  
**Authors:** Colin Raffel, Noam Shazeer, Adam Roberts, Katherine Lee, Sharan Narang, Michael Matena, Yanqi Zhou, Wei Li, Peter J. Liu  
**Year:** 2019 (published in JMLR 2020)

## Overview

This notebook implements the core concepts of T5 from scratch in PyTorch:
1. A small encoder-decoder Transformer with multi-head attention and relative position bias
2. Span corruption pre-training objective
3. Text-to-text format for multiple NLP tasks (classification, summarization, QA)
4. Multi-task fine-tuning with a single model

The key insight: **every NLP task is cast as text-in → text-out**, using task prefixes like `"classify sentiment:"`, `"summarize:"`, `"question:"`.

## 1. Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import re
from collections import Counter, defaultdict
import math

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## 2. Vocabulary & Tokenizer

A simple whitespace + punctuation tokenizer that builds a word-level vocabulary.
The paper uses SentencePiece (32K subword vocabulary); we use word-level for simplicity.

In [2]:
class SimpleTokenizer:
    """Minimal word-level tokenizer with special tokens and sentinels."""
    def __init__(self, num_sentinels=10):
        self.num_sentinels = num_sentinels
        self.special_tokens = ['<pad>', '<sos>', '<eos>', '<unk>']
        self.sentinel_tokens = [f'<extra_id_{i}>' for i in range(num_sentinels)]
        self.vocab = {}
        self.id_to_token = {}
        self.fitted = False

    def _tokenize(self, text):
        # Split on whitespace and punctuation, keep punctuation as separate tokens
        tokens = re.findall(r"\w+|[!?.,;:'\-\"/]|<extra_id_\d+>", text.lower())
        return tokens

    def fit(self, texts):
        all_tokens = set(self.special_tokens + self.sentinel_tokens)
        for text in texts:
            all_tokens.update(self._tokenize(text))
        self.vocab = {tok: idx for idx, tok in enumerate(sorted(all_tokens))}
        self.id_to_token = {idx: tok for tok, idx in self.vocab.items()}
        self.fitted = True
        return self

    def encode(self, text):
        tokens = self._tokenize(text)
        ids = [self.vocab.get(t, self.vocab['<unk>']) for t in tokens]
        return ids

    def decode(self, ids):
        tokens = [self.id_to_token.get(i, '<unk>') for i in ids]
        # Skip special tokens in output
        text = ' '.join(t for t in tokens if t not in ['<pad>', '<sos>'])
        # Clean up spacing around punctuation
        text = re.sub(r'\s+([!?.,;:])', r'\1', text)
        text = text.replace(' <eos>', '').replace('<eos>', '')
        return text.strip()

    @property
    def pad_id(self): return self.vocab['<pad>']
    @property
    def sos_id(self): return self.vocab['<sos>']
    @property
    def eos_id(self): return self.vocab['<eos>']
    @property
    def unk_id(self): return self.vocab['<unk>']
    @property
    def vocab_size(self): return len(self.vocab)

    def sentinel_id(self, k):
        return self.vocab[f'<extra_id_{k}>']


## 3. Transformer Building Blocks

### 3a. Multi-Head Attention with Relative Position Bias

T5 uses a simplified form of relative position embeddings: scalar biases added to the
attention logits based on the relative distance between query and key positions.

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_dist=128, is_causal=False, has_cross_attention=False):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.is_causal = is_causal
        self.has_cross_attention = has_cross_attention

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        # Relative position bias: one scalar per (distance_bucket, head)
        # T5 uses buckets; we use a simpler linear distance with clamping
        self.max_dist = max_dist
        self.rel_pos_bias = nn.Embedding(2 * max_dist + 1, n_heads)

    def _relative_position_bucket(self, seq_len, device):
        """Create relative position bias matrix."""
        context_pos = torch.arange(seq_len, device=device)[:, None]
        memory_pos = torch.arange(seq_len, device=device)[None, :]
        relative_pos = memory_pos - context_pos  # (seq_len, seq_len)
        # Clamp to [-max_dist, max_dist] and shift to [0, 2*max_dist]
        relative_pos = relative_pos.clamp(-self.max_dist, self.max_dist) + self.max_dist
        return relative_pos  # (seq_len, seq_len)

    def forward(self, x, context=None, mask=None):
        """
        x: (batch, seq_len, d_model) - query input
        context: (batch, ctx_len, d_model) - for cross-attention (None for self-attention)
        mask: (batch, ctx_len) padding mask (1=attend, 0=pad)
        """
        batch_size, seq_len, _ = x.shape
        ctx = context if context is not None else x
        ctx_len = ctx.shape[1]

        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_proj(ctx).view(batch_size, ctx_len, self.n_heads, self.d_k).transpose(1, 2)
        v = self.v_proj(ctx).view(batch_size, ctx_len, self.n_heads, self.d_k).transpose(1, 2)

        # Attention scores: (batch, n_heads, seq_len, ctx_len)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Add relative position bias for self-attention
        if not self.has_cross_attention:
            rel_pos = self._relative_position_bucket(seq_len, x.device)
            rel_bias = self.rel_pos_bias(rel_pos)  # (seq_len, seq_len, n_heads)
            rel_bias = rel_bias.permute(2, 0, 1).unsqueeze(0)  # (1, n_heads, seq_len, seq_len)
            scores = scores + rel_bias

        # Apply causal mask for decoder self-attention
        if self.is_causal:
            causal_mask = torch.triu(torch.ones(seq_len, ctx_len, device=x.device), diagonal=1).bool()
            scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        # Apply padding mask: mask is (batch, ctx_len) -> (batch, 1, 1, ctx_len)
        if mask is not None:
            key_mask = mask.unsqueeze(1).unsqueeze(1)  # (batch, 1, 1, ctx_len)
            scores = scores.masked_fill(key_mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, v)  # (batch, n_heads, seq_len, d_k)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.out_proj(attn_output)


### 3b. Feed-Forward Network & Layer Normalization

In [4]:
class RMSNorm(nn.Module):
    """Simplified layer normalization (T5 style: rescale only, no additive bias)."""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight


class PositionwiseFFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.w2(F.relu(self.w1(x)))


### 3c. Encoder & Decoder Blocks

In [5]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, max_dist=128):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, max_dist, is_causal=False)
        self.ffn = PositionwiseFFN(d_model, d_ff)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        # Pre-LN: normalize before sublayer
        normed = self.norm1(x)
        x = x + self.dropout(self.self_attn(normed, mask=src_mask))
        normed = self.norm2(x)
        x = x + self.dropout(self.ffn(normed))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, max_dist=128):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, max_dist, is_causal=True)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, max_dist, has_cross_attention=True)
        self.ffn = PositionwiseFFN(d_model, d_ff)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.norm3 = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        normed = self.norm1(x)
        x = x + self.dropout(self.self_attn(normed, mask=tgt_mask))
        normed = self.norm2(x)
        x = x + self.dropout(self.cross_attn(normed, context=enc_output, mask=src_mask))
        normed = self.norm3(x)
        x = x + self.dropout(self.ffn(normed))
        return x


### 3d. Full T5 Model

In [6]:
class T5Model(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, d_ff=512,
                 n_enc_layers=2, n_dec_layers=2, dropout=0.1, max_dist=128):
        super().__init__()
        self.d_model = d_model
        # Shared embedding for encoder input, decoder input, and output projection
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout, max_dist)
            for _ in range(n_enc_layers)
        ])
        self.decoder = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout, max_dist)
            for _ in range(n_dec_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        # Output projection shares weights with embedding
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight  # weight tying

        self.dropout = nn.Dropout(dropout)

    def encode(self, src_ids, src_mask=None):
        """Encode source sequence."""
        x = self.dropout(self.embedding(src_ids) * (self.d_model ** 0.5))
        for block in self.encoder:
            x = block(x, src_mask)
        return x

    def decode(self, tgt_ids, enc_output, src_mask=None, tgt_mask=None):
        """Decode target sequence given encoder output."""
        x = self.dropout(self.embedding(tgt_ids) * (self.d_model ** 0.5))
        for block in self.decoder:
            x = block(x, enc_output, src_mask, tgt_mask)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits

    def forward(self, src_ids, tgt_ids, src_mask=None, tgt_mask=None):
        enc_output = self.encode(src_ids, src_mask)
        logits = self.decode(tgt_ids, enc_output, src_mask, tgt_mask)
        return logits


## 4. Span Corruption Pre-training Objective

T5's pre-training task: corrupt ~15% of tokens in contiguous spans, replace each span
with a unique sentinel token, and train the model to reconstruct the original spans.

**Example:**
- Original: `The quick brown fox jumps`
- Corrupted: `The quick <extra_id_0> fox <extra_id_1>`
- Target: `<extra_id_0> brown <extra_id_1> jumps <eos>`

In [7]:
def corrupt_spans(token_ids, tokenizer, corruption_rate=0.15, mean_span_length=3):
    """
    T5-style span corruption.

    Returns (corrupted_input_ids, target_ids) where:
    - corrupted_input_ids: original with spans replaced by sentinel tokens
    - target_ids: <sos> <extra_id_0> span0_tokens <extra_id_1> span1_tokens ... <eos>
    """
    seq_len = len(token_ids)
    num_to_corrupt = max(1, int(seq_len * corruption_rate))
    # Number of spans
    num_spans = max(1, num_to_corrupt // mean_span_length)

    # Randomly choose span start positions
    # Simple approach: pick random positions and group into spans
    all_positions = list(range(seq_len))
    random.shuffle(all_positions)
    corrupt_positions = sorted(all_positions[:num_to_corrupt])

    # Group consecutive positions into spans
    spans = []
    current_span = [corrupt_positions[0]]
    for pos in corrupt_positions[1:]:
        if pos == current_span[-1] + 1:
            current_span.append(pos)
        else:
            spans.append(current_span)
            current_span = [pos]
    spans.append(current_span)

    # Limit number of spans to num_sentinels
    if len(spans) > tokenizer.num_sentinels:
        # Merge excess spans
        while len(spans) > tokenizer.num_sentinels:
            spans[-2] = spans[-2] + spans[-1]
            spans.pop()

    # Build corrupted input and target
    corrupted = []
    target = [tokenizer.sos_id]
    span_set = set()
    for span in spans:
        span_set.update(span)

    sentinel_idx = 0
    pos = 0
    while pos < seq_len:
        if pos in span_set:
            # Find which span this belongs to
            for si, span in enumerate(spans):
                if pos in span:
                    # Add sentinel to corrupted input
                    corrupted.append(tokenizer.sentinel_id(si))
                    # Add sentinel + original span tokens to target
                    target.append(tokenizer.sentinel_id(si))
                    for sp in span:
                        target.append(token_ids[sp])
                    pos = span[-1] + 1
                    break
            else:
                corrupted.append(token_ids[pos])
                pos += 1
        else:
            corrupted.append(token_ids[pos])
            pos += 1

    target.append(tokenizer.eos_id)
    return corrupted, target


## 5. Pre-training Data (Synthetic Corpus)

In [8]:
# Generate a small synthetic English corpus for pre-training
corpus_sentences = [
    "the cat sat on the mat and looked at the sun",
    "a dog ran through the park chasing a small ball",
    "the sun rises in the east and sets in the west",
    "she opened the book and read the first chapter carefully",
    "the train arrived at the station on time every day",
    "children played in the garden while their parents watched",
    "the scientist conducted an experiment in the laboratory",
    "a bird flew across the sky and landed on a tree",
    "the river flowed quietly through the green valley",
    "he bought fresh bread from the bakery on main street",
    "the teacher explained the lesson to the attentive students",
    "music played softly in the background during the dinner",
    "the old man walked slowly along the beach at sunset",
    "flowers bloomed in the garden during the warm spring",
    "the chef prepared a delicious meal for the guests",
    "rain fell gently on the roof throughout the night",
    "the library was quiet and filled with many old books",
    "a young girl painted a picture of her favorite pet",
    "the city was busy with people rushing to work",
    "mountains stood tall against the clear blue sky",
    "she wrote a letter to her friend who lived far away",
    "the clock on the wall ticked loudly in the silence",
    "fish swam in the clear water of the small pond",
    "the boy kicked the ball high into the air and laughed",
    "a gentle breeze blew through the open window",
    "the store sold fruits and vegetables at reasonable prices",
    "stars twinkled brightly in the dark night sky",
    "the doctor told the patient to rest and drink water",
    "they walked together along the path through the forest",
    "the movie was exciting and kept everyone on the edge",
    "she planted seeds in the soil and watered them daily",
    "the bridge crossed over the wide river near the town",
    "he fixed the broken chair with some nails and glue",
    "the children built a sandcastle on the beach together",
    "she sang a beautiful song that moved everyone to tears",
    "the boat sailed across the lake on a calm morning",
    "he studied hard for the exam and passed with honors",
    "the garden was full of colorful butterflies in summer",
    "they enjoyed a picnic under the big oak tree",
    "the artist drew a sketch of the mountain landscape",
    "snow covered the ground and the trees in winter",
    "she made a cake for her mother on her birthday",
    "the wind howled through the trees during the storm",
    "he drove his car along the winding mountain road",
    "the baby laughed when the puppy licked her face",
    "students raised their hands to answer the question",
    "the moon shone brightly over the calm ocean waves",
    "she found a shell on the beach and put it in her pocket",
    "the restaurant served pasta and pizza to many customers",
]

# Repeat and vary slightly to get more data
pretrain_texts = []
for s in corpus_sentences:
    pretrain_texts.append(s)
    # Also add some concatenated versions for longer sequences
for i in range(0, len(corpus_sentences) - 1, 2):
    pretrain_texts.append(corpus_sentences[i] + " " + corpus_sentences[i + 1])

print(f"Pre-training corpus: {len(pretrain_texts)} sentences")

# Build tokenizer from corpus
tokenizer = SimpleTokenizer(num_sentinels=10)
tokenizer.fit(pretrain_texts)
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Sample tokens: {list(tokenizer.vocab.items())[:10]}")


Pre-training corpus: 73 sentences
Vocabulary size: 281
Sample tokens: [('<eos>', 0), ('<extra_id_0>', 1), ('<extra_id_1>', 2), ('<extra_id_2>', 3), ('<extra_id_3>', 4), ('<extra_id_4>', 5), ('<extra_id_5>', 6), ('<extra_id_6>', 7), ('<extra_id_7>', 8), ('<extra_id_8>', 9)]


## 6. Pre-training Loop

In [9]:
def make_batch(sequences, tokenizer, max_len=64):
    """Pad a list of token-id sequences into a batch tensor with masks."""
    batch_size = len(sequences)
    padded = torch.zeros(batch_size, max_len, dtype=torch.long) + tokenizer.pad_id
    mask = torch.zeros(batch_size, max_len, dtype=torch.long)
    for i, seq in enumerate(sequences):
        length = min(len(seq), max_len)
        padded[i, :length] = torch.tensor(seq[:length], dtype=torch.long)
        mask[i, :length] = 1
    return padded, mask


def create_pretraining_batch(texts, tokenizer, batch_size=32, max_len=64):
    """Create a batch of span-corrupted examples."""
    inputs, targets = [], []
    for text in texts:
        token_ids = tokenizer.encode(text)
        if len(token_ids) < 5:
            continue
        corrupted, target = corrupt_spans(token_ids, tokenizer)
        inputs.append(corrupted)
        targets.append(target)

    # Pad and create tensors
    src_padded, src_mask = make_batch(inputs, tokenizer, max_len)
    # For decoder: shift target right (prepend sos, drop last)
    # target already starts with sos, so we use it as decoder input
    # and the shifted version (drop sos, keep rest) as labels
    tgt_padded, tgt_mask = make_batch(targets, tokenizer, max_len)

    # Decoder input: all tokens except last
    dec_input = tgt_padded[:, :-1]
    dec_mask = tgt_mask[:, :-1]
    # Labels: all tokens except first (sos)
    labels = tgt_padded[:, 1:]

    return src_padded, src_mask, dec_input, dec_mask, labels


# Initialize model
model = T5Model(
    vocab_size=tokenizer.vocab_size,
    d_model=128,
    n_heads=4,
    d_ff=512,
    n_enc_layers=2,
    n_dec_layers=2,
    dropout=0.1,
    max_dist=64
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")


Model parameters: 957,976


In [10]:
# Pre-training loop
EPOCHS = 20
LR = 0.001
WARMUP_STEPS = 50

optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9)
loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id)

step = 0
total_steps = EPOCHS * (len(pretrain_texts) // 32 + 1)
pretrain_losses = []

for epoch in range(EPOCHS):
    random.shuffle(pretrain_texts)
    epoch_loss = 0
    n_batches = 0

    for i in range(0, len(pretrain_texts), 32):
        batch_texts = pretrain_texts[i:i+32]
        if len(batch_texts) < 2:
            continue

        src_ids, src_mask, dec_in, dec_mask, labels = create_pretraining_batch(
            batch_texts, tokenizer, batch_size=len(batch_texts), max_len=64
        )
        src_ids = src_ids.to(device)
        src_mask = src_mask.to(device)
        dec_in = dec_in.to(device)
        dec_mask = dec_mask.to(device)
        labels = labels.to(device)

        # Learning rate schedule: linear warmup then inverse sqrt
        step += 1
        if step < WARMUP_STEPS:
            lr_scale = step / WARMUP_STEPS
        else:
            lr_scale = (WARMUP_STEPS / step) ** 0.5
        for pg in optimizer.param_groups:
            pg['lr'] = LR * lr_scale

        logits = model(src_ids, dec_in, src_mask, dec_mask)
        loss = loss_fn(logits.reshape(-1, tokenizer.vocab_size), labels.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / max(n_batches, 1)
    pretrain_losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - avg loss: {avg_loss:.4f}")

print(f"\nPre-training complete. Final loss: {pretrain_losses[-1]:.4f}")


Epoch 5/20 - avg loss: 97.7974


Epoch 10/20 - avg loss: 44.9655


Epoch 15/20 - avg loss: 15.4738


Epoch 20/20 - avg loss: 9.3094

Pre-training complete. Final loss: 9.3094


In [11]:
# Visualize pre-training loss
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(pretrain_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('T5 Pre-training Loss (Span Corruption)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pretraining_loss.png', dpi=100)
plt.show()
print("Loss curve saved.")


Loss curve saved.


## 7. Text-to-Text Downstream Tasks

We create three toy NLP tasks, all formatted as text → text:
1. **Sentiment Classification:** `"classify sentiment: <text>"` → `"positive"`/`"negative"`
2. **Summarization:** `"summarize: <paragraph>"` → `"<one-sentence summary>"`
3. **Question Answering:** `"question: <q> context: <context>"` → `"<answer>"`

In [12]:
# --- Task 1: Sentiment Classification ---
positive_templates = [
    "the movie was absolutely wonderful and amazing",
    "i really enjoyed the food at this restaurant",
    "what a fantastic day it has been for everyone",
    "the concert was incredible and the music was great",
    "she loved the gift her husband gave her",
    "the vacation was the best experience of my life",
    "the book was fascinating and i could not put it down",
    "we had a delightful time at the park today",
    "the service at the hotel was excellent and friendly",
    "the performance was brilliant and deeply moving",
    "this is the most beautiful painting i have ever seen",
    "the weather was perfect for a picnic in the garden",
    "he felt very happy about his new job and colleagues",
    "the cake tasted delicious and was beautifully decorated",
    "the team played wonderfully and won the match easily",
]

negative_templates = [
    "the movie was terrible and a complete waste of time",
    "the food at the restaurant was cold and tasteless",
    "what a horrible day it has been for everyone here",
    "the concert was boring and the sound was very bad",
    "she hated the gift and threw it away immediately",
    "the vacation was the worst experience of my life",
    "the book was dull and i could not finish reading it",
    "we had a miserable time at the park in the rain",
    "the service at the hotel was rude and very slow",
    "the performance was awful and completely uninspiring",
    "this is the ugliest painting i have ever seen",
    "the weather was awful for our picnic in the garden",
    "he felt very sad about losing his old job suddenly",
    "the cake tasted awful and was poorly decorated",
    "the team played terribly and lost the match badly",
]

sentiment_data = []
for text in positive_templates:
    sentiment_data.append((f"classify sentiment: {text}", "positive"))
for text in negative_templates:
    sentiment_data.append((f"classify sentiment: {text}", "negative"))

print(f"Sentiment examples: {len(sentiment_data)}")
print(f"  Example: {sentiment_data[0]}")


Sentiment examples: 30
  Example: ('classify sentiment: the movie was absolutely wonderful and amazing', 'positive')


In [13]:
# --- Task 2: Summarization ---
# Each example: a "paragraph" (2-3 sentences) -> a one-sentence summary
summarization_data = [
    ("summarize: the cat sat on the mat. it looked at the sun. the sun was warm and bright.",
     "the cat enjoyed the warm sun on the mat"),
    ("summarize: the dog ran through the park. it chased a ball. the ball was red and small.",
     "the dog chased a red ball in the park"),
    ("summarize: she opened the book. she read the first chapter. the chapter was about history.",
     "she read a history chapter in a book"),
    ("summarize: the train arrived at the station. it was on time. many people got on the train.",
     "the train arrived on time and people boarded"),
    ("summarize: children played in the garden. their parents watched them. everyone was happy.",
     "children played happily while parents watched"),
    ("summarize: the scientist worked in the lab. she conducted an experiment. the results were interesting.",
     "the scientist ran an interesting lab experiment"),
    ("summarize: a bird flew across the sky. it landed on a tree. the tree was tall and green.",
     "a bird flew and landed on a tall tree"),
    ("summarize: the river flowed through the valley. the valley was green. the water was clear.",
     "the river flowed through a green clear valley"),
    ("summarize: he bought bread from the bakery. the bread was fresh. it smelled wonderful.",
     "he bought fresh bread from the bakery"),
    ("summarize: the teacher explained the lesson. the students listened carefully. they asked good questions.",
     "the teacher gave a lesson to attentive students"),
    ("summarize: rain fell on the roof all night. the sound was gentle. everyone slept peacefully.",
     "gentle rain helped everyone sleep peacefully"),
    ("summarize: the library was quiet. it had many old books. she loved reading there.",
     "she enjoyed reading in the quiet library"),
    ("summarize: the city was busy. people rushed to work. cars filled the streets.",
     "the busy city was full of rushing people"),
    ("summarize: stars twinkled in the sky. the night was dark. the moon was bright.",
     "stars and a bright moon lit the dark night"),
    ("summarize: the chef cooked a meal. it was delicious. the guests were very happy.",
     "the chef made a delicious meal for guests"),
]

print(f"Summarization examples: {len(summarization_data)}")
print(f"  Example: {summarization_data[0]}")


Summarization examples: 15
  Example: ('summarize: the cat sat on the mat. it looked at the sun. the sun was warm and bright.', 'the cat enjoyed the warm sun on the mat')


In [14]:
# --- Task 3: Question Answering ---
qa_data = [
    ("question: what color is the sky context: the sky is blue on a clear day",
     "blue"),
    ("question: where does the sun rise context: the sun rises in the east every morning",
     "east"),
    ("question: what animal barks context: a dog is an animal that barks and runs",
     "dog"),
    ("question: what do bees make context: bees make honey from flowers in the garden",
     "honey"),
    ("question: how many legs does a spider have context: a spider has eight legs and weaves webs",
     "eight"),
    ("question: what is the capital of france context: paris is the capital city of france in europe",
     "paris"),
    ("question: what season comes after winter context: spring comes after winter and brings flowers",
     "spring"),
    ("question: what do plants need to grow context: plants need water and sunlight to grow tall",
     "water and sunlight"),
    ("question: what is the largest planet context: jupiter is the largest planet in our solar system",
     "jupiter"),
    ("question: who wrote romeo and juliet context: shakespeare wrote romeo and juliet long ago",
     "shakespeare"),
    ("question: what is ice context: ice is frozen water that forms when it is cold",
     "frozen water"),
    ("question: what color are leaves context: most leaves are green because of chlorophyll",
     "green"),
    ("question: where do fish live context: fish live in water and swim with their fins",
     "water"),
    ("question: what do birds use to fly context: birds use their wings to fly high in the sky",
     "wings"),
    ("question: what is snow context: snow is frozen rain that falls in winter months",
     "frozen rain"),
]

print(f"QA examples: {len(qa_data)}")
print(f"  Example: {qa_data[0]}")


QA examples: 15
  Example: ('question: what color is the sky context: the sky is blue on a clear day', 'blue')


## 8. Multi-task Fine-tuning

In [15]:
# Combine all tasks into one training set
all_task_data = sentiment_data + summarization_data + qa_data
random.shuffle(all_task_data)

# Add task labels to tokenizer vocabulary
all_texts_for_vocab = [inp for inp, _ in all_task_data] + [out for _, out in all_task_data]
# Refit tokenizer with all texts (pretrain + downstream)
tokenizer = SimpleTokenizer(num_sentinels=10)
tokenizer.fit(pretrain_texts + all_texts_for_vocab)
print(f"Extended vocabulary size: {tokenizer.vocab_size}")

# Rebuild model with extended vocab
model = T5Model(
    vocab_size=tokenizer.vocab_size,
    d_model=128,
    n_heads=4,
    d_ff=512,
    n_enc_layers=2,
    n_dec_layers=2,
    dropout=0.1,
    max_dist=64
).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

# Quick pre-train on the corpus to give the model some language understanding
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.98), eps=1e-9)
loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id)

print("Quick pre-training (10 epochs)...")
for epoch in range(10):
    random.shuffle(pretrain_texts)
    for i in range(0, len(pretrain_texts), 32):
        batch_texts = pretrain_texts[i:i+32]
        if len(batch_texts) < 2:
            continue
        src_ids, src_mask, dec_in, dec_mask, labels = create_pretraining_batch(
            batch_texts, tokenizer, batch_size=len(batch_texts), max_len=64
        )
        src_ids, src_mask = src_ids.to(device), src_mask.to(device)
        dec_in, dec_mask = dec_in.to(device), dec_mask.to(device)
        labels = labels.to(device)

        logits = model(src_ids, dec_in, src_mask, dec_mask)
        loss = loss_fn(logits.reshape(-1, tokenizer.vocab_size), labels.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

print("Pre-training done. Starting fine-tuning...")


Extended vocabulary size: 466
Model parameters: 981,656
Quick pre-training (10 epochs)...


Pre-training done. Starting fine-tuning...


In [16]:
# Fine-tuning on all tasks (multi-task)
FINETUNE_EPOCHS = 40
ft_optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, betas=(0.9, 0.98), eps=1e-9)

def create_task_batch(task_data, tokenizer, batch_size=16, max_len=80):
    """Create batch for text-to-text tasks."""
    inputs, targets = [], []
    for inp_text, tgt_text in task_data:
        inp_ids = [tokenizer.sos_id] + tokenizer.encode(inp_text)
        tgt_ids = [tokenizer.sos_id] + tokenizer.encode(tgt_text) + [tokenizer.eos_id]
        inputs.append(inp_ids)
        targets.append(tgt_ids)

    src_padded, src_mask = make_batch(inputs, tokenizer, max_len)
    tgt_padded, tgt_mask = make_batch(targets, tokenizer, max_len)

    dec_input = tgt_padded[:, :-1]
    dec_mask = tgt_mask[:, :-1]
    labels = tgt_padded[:, 1:]

    return src_padded, src_mask, dec_input, dec_mask, labels


ft_losses = []
for epoch in range(FINETUNE_EPOCHS):
    random.shuffle(all_task_data)
    epoch_loss = 0
    n_batches = 0

    for i in range(0, len(all_task_data), 16):
        batch = all_task_data[i:i+16]
        if len(batch) < 2:
            continue

        src_ids, src_mask, dec_in, dec_mask, labels = create_task_batch(
            batch, tokenizer, batch_size=len(batch), max_len=80
        )
        src_ids, src_mask = src_ids.to(device), src_mask.to(device)
        dec_in, dec_mask = dec_in.to(device), dec_mask.to(device)
        labels = labels.to(device)

        logits = model(src_ids, dec_in, src_mask, dec_mask)
        loss = loss_fn(logits.reshape(-1, tokenizer.vocab_size), labels.reshape(-1))

        ft_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        ft_optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / max(n_batches, 1)
    ft_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Fine-tune epoch {epoch+1}/{FINETUNE_EPOCHS} - avg loss: {avg_loss:.4f}")

print(f"\nFine-tuning complete. Final loss: {ft_losses[-1]:.4f}")


Fine-tune epoch 10/40 - avg loss: 9.6420


Fine-tune epoch 20/40 - avg loss: 6.1703


Fine-tune epoch 30/40 - avg loss: 4.2061


Fine-tune epoch 40/40 - avg loss: 3.4674

Fine-tuning complete. Final loss: 3.4674


In [17]:
# Plot fine-tuning loss
plt.figure(figsize=(8, 4))
plt.plot(ft_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('T5 Multi-task Fine-tuning Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('finetuning_loss.png', dpi=100)
plt.show()
print("Fine-tuning loss curve saved.")


Fine-tuning loss curve saved.


## 9. Inference & Demonstration

In [18]:
def generate(model, tokenizer, src_text, max_len=40, device='cpu'):
    """Greedy decoding for inference."""
    model.eval()
    with torch.no_grad():
        src_ids = torch.tensor([[tokenizer.sos_id] + tokenizer.encode(src_text)], device=device)
        src_mask = torch.ones_like(src_ids)

        # Encode source
        enc_output = model.encode(src_ids, src_mask)

        # Start decoding with sos
        tgt_ids = [tokenizer.sos_id]

        for _ in range(max_len):
            tgt_tensor = torch.tensor([tgt_ids], device=device)
            tgt_mask = torch.ones_like(tgt_tensor)

            logits = model.decode(tgt_tensor, enc_output, src_mask, tgt_mask)
            next_token = logits[0, -1].argmax().item()

            if next_token == tokenizer.eos_id:
                break
            tgt_ids.append(next_token)

        return tokenizer.decode(tgt_ids[1:])  # skip sos


# Test on sentiment classification
print("=" * 60)
print("TASK 1: Sentiment Classification")
print("=" * 60)
test_sentiments = [
    "classify sentiment: the movie was absolutely wonderful and amazing",
    "classify sentiment: the food at the restaurant was cold and tasteless",
    "classify sentiment: the vacation was the best experience of my life",
    "classify sentiment: the book was dull and i could not finish reading it",
]
for text in test_sentiments:
    result = generate(model, tokenizer, text, max_len=10, device=device)
    print(f"  Input: {text}")
    print(f"  Output: {result}")
    print()


TASK 1: Sentiment Classification
  Input: classify sentiment: the movie was absolutely wonderful and amazing
  Output: positive

  Input: classify sentiment: the food at the restaurant was cold and tasteless
  Output: positive

  Input: classify sentiment: the vacation was the best experience of my life
  Output: positive

  Input: classify sentiment: the book was dull and i could not finish reading it
  Output: positive



In [19]:
# Test on summarization
print("=" * 60)
print("TASK 2: Summarization")
print("=" * 60)
test_summaries = [
    "summarize: the cat sat on the mat. it looked at the sun. the sun was warm and bright.",
    "summarize: the dog ran through the park. it chased a ball. the ball was red and small.",
    "summarize: the train arrived at the station. it was on time. many people got on the train.",
]
for text in test_summaries:
    result = generate(model, tokenizer, text, max_len=20, device=device)
    print(f"  Input: {text}")
    print(f"  Output: {result}")
    print()


TASK 2: Summarization


  Input: summarize: the cat sat on the mat. it looked at the sun. the sun was warm and bright.
  Output: the the the the the the the the the the the the the the the the the the the the

  Input: summarize: the dog ran through the park. it chased a ball. the ball was red and small.
  Output: the dog of the dog of the dog of the dog

  Input: summarize: the train arrived at the station. it was on time. many people got on the train.
  Output: the city sun



In [20]:
# Test on question answering
print("=" * 60)
print("TASK 3: Question Answering")
print("=" * 60)
test_qa = [
    "question: what color is the sky context: the sky is blue on a clear day",
    "question: where does the sun rise context: the sun rises in the east every morning",
    "question: what animal barks context: a dog is an animal that barks and runs",
    "question: what do bees make context: bees make honey from flowers in the garden",
]
for text in test_qa:
    result = generate(model, tokenizer, text, max_len=10, device=device)
    print(f"  Input: {text}")
    print(f"  Output: {result}")
    print()


TASK 3: Question Answering
  Input: question: what color is the sky context: the sky is blue on a clear day
  Output: jupiter

  Input: question: where does the sun rise context: the sun rises in the east every morning
  Output: jupiter

  Input: question: what animal barks context: a dog is an animal that barks and runs
  Output: dog

  Input: question: what do bees make context: bees make honey from flowers in the garden
  Output: paris



## 10. Summary

This notebook demonstrated the key concepts of T5:

1. **Text-to-text format:** All three tasks (sentiment, summarization, QA) use the same model, same loss function, and same training procedure — only the task prefix in the input differs.
2. **Span corruption pre-training:** The model was pre-trained to reconstruct corrupted spans, learning general text representations before task-specific fine-tuning.
3. **Encoder-decoder Transformer:** Bidirectional encoder + autoregressive decoder with relative position bias and shared input/output embeddings.
4. **Multi-task fine-tuning:** A single model was fine-tuned on all three tasks simultaneously, demonstrating the unified framework.

The paper's key insight — that casting all NLP tasks as text-in/text-out unifies the entire pipeline — became foundational for modern instruction-tuned LLMs.

**Paper:** [Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer](https://arxiv.org/abs/1910.10683)

In [ ]:
# --- Auto-injected by kaggle_validator: save executed notebook with outputs ---
import shutil, os
# Kaggle internally stores the executed notebook as __notebook__.ipynb
if os.path.exists('/kaggle/working/__notebook__.ipynb'):
    shutil.copy('/kaggle/working/__notebook__.ipynb', '/kaggle/working/executed_solution.ipynb')
    print('Executed notebook saved to output folder.')
else:
    # Fallback: try alternative path
    import glob
    candidates = glob.glob('/kaggle/working/*.ipynb')
    print(f'Available notebooks in output: {candidates}')
